# Code to analyse output BAHA5P, before (first fit) and after adjustig to max stable gain

28 February 2021 Guido Cattani

Is it worth to try to set gain to maximal stable values?

In [1]:
from pathlib import Path
import pandas as pd
from scipy.stats import wilcoxon as wilcoxon

In [2]:
# read output force levels measured with a 65 dB ISDS input signal
def read_BCD_output_65():
    f_in = '/media/guido/LACIE/Cingle_Guido/Master/BCD_band_output.xlsx'
    p_in = Path(f_in)   
    df = pd.read_excel(p_in, header=0, nrows=85)
    df = df.drop(['Unnamed: 0'], axis=1)
    df.set_index('Study_ID', inplace=True)
    df = df.fillna(pd.NA)
    return df

In [3]:
def select_bh5(df):
    # select BAHA5P data
    is_baha5p =  df['Device']=='BAHA5P'
    df_baha5p = df[is_baha5p]
    df_baha5p.pop('Device')
    return(df_baha5p)

In [4]:
# read output force levels measured with a 65 dB ISDS input signal
def read_BAHA5P_maximal_output_65():
    f_in = '/media/guido/LACIE/Cingle_Guido/Master/BAHA5P_band_maximal_output_65.xlsx'
    p_in = Path(f_in)   
    df = pd.read_excel(p_in, header=0, nrows=35)
    df = df.drop(['Unnamed: 0', 'Device'], axis=1)
    df.set_index('Study_ID', inplace=True)
    df = df.fillna(pd.NA)
    return df

In [5]:
def diff_output():
    bcd_fit = read_BCD_output_65()
    bh5_fit = select_bh5(bcd_fit)
    bh5_max = read_BAHA5P_maximal_output_65()
    diff = bh5_max - bh5_fit
    diff = diff.dropna()
    return diff

In [6]:
diff = diff_output()

In [7]:
len(diff)

35

In [8]:
diff

,250_Hz,315_Hz,400_Hz,500_Hz,630_Hz,800_Hz,1000_Hz,1250_Hz,1600_Hz,2000_Hz,2500_Hz,3150_Hz,4000_Hz,5000_Hz,6300_Hz,8000_Hz
Study_ID,,,,,,,,,,,,,,,,
21,-1.2,-3.1,1.3,1.3,0.9,0.8,0.7,1.0,1.6,2.4,1.3,0.7,0.8,1.2,1.5,1.2
48,3.1,2.7,1.9,2.2,2.2,2.4,2.6,2.1,1.3,0.7,0.1,-0.1,0.1,1.2,2.2,2.8
52,4.7,9.2,7.4,6.2,6.3,5.7,4.2,1.6,0.2,-0.2,-0.4,-0.5,-0.7,-0.3,-0.1,0.3
54,-0.3,-2.8,-3.5,-1.6,-1.2,-0.5,1.6,6.6,8.9,7.2,3.0,1.1,-0.1,-0.7,-0.9,-0.7
55,0.7,1.5,0.0,-0.9,-0.6,-0.4,-0.3,0.1,1.9,3.9,2.4,0.6,-0.8,-1.2,-1.0,-0.8
56,4.7,5.1,4.0,3.2,3.4,3.7,3.5,3.4,2.1,0.7,-0.2,-0.5,-1.4,-1.9,-1.7,-1.6
57,0.0,0.2,0.2,-0.1,-0.2,-0.1,0.0,-0.2,0.0,0.4,0.2,0.0,-0.1,-0.2,-0.1,-0.1
58,0.0,1.1,1.5,0.9,0.8,0.6,0.5,0.5,1.5,3.2,2.3,1.0,0.5,0.2,-0.2,-0.4
59,-0.7,-1.1,0.1,0.4,0.5,0.6,0.8,2.9,4.0,3.8,1.3,0.1,0.2,0.3,0.3,0.3


In [9]:
mdn = diff.median()

In [10]:
quantiles = [0.10, 0.50, 0.90]
q = diff.quantile(q=quantiles)
q

,250_Hz,315_Hz,400_Hz,500_Hz,630_Hz,800_Hz,1000_Hz,1250_Hz,1600_Hz,2000_Hz,2500_Hz,3150_Hz,4000_Hz,5000_Hz,6300_Hz,8000_Hz
0.1,-0.22,-0.52,-1.08,-0.74,-0.48,-0.36,-0.30,-0.20,0.14,0.14,-0.10,-0.36,-0.86,-1.12,-1.18,-1.26
0.5,0.20,0.70,0.40,0.40,0.40,0.40,0.70,1.30,1.90,2.90,1.70,0.60,0.10,0.00,-0.20,-0.20
0.9,3.46,3.32,3.18,3.12,3.40,3.76,3.46,3.94,5.58,6.96,5.62,3.82,1.04,0.86,1.06,1.10


In [11]:
# make list with column names (frequencies)

clmns = diff.columns
l = list()
for clm in clmns:
    l.append(clm)

In [12]:
''' Compare output of BAHA5P at 65 dB, first fit versus maxiamal stable
The Wilcoxon signed-rank test tests the null hypothesis that 
two related paired samples come from the same distribution. 
In particular, it tests whether the distribution of the differences x - y is symmetric 
about zero. It is a non-parametric version of the paired T-test.'''

wlcx = dict()

for i in range(0, 16):
    x = diff.iloc[:, i]
    md = mdn[i]
    f = l[i]
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html
    statistic, pVal = wilcoxon(x)
    statistic = round(statistic, 4)
    pVal = round(pVal, 4)
    pVal_promil=pVal*100
    st = {f: [md, statistic, pVal, pVal_promil]}
    wlcx.update(st)
wilcoxon_test = pd.DataFrame.from_dict(wlcx, dtype='float')
idx = {0 : 'Median of difference', 1 : 'Wilcoxon signed rank statistic', 2 : 'p value', 3 : 'p value %'}
wilcoxon_test.rename(index=idx, inplace=True)

In [13]:
wlcx

{'250_Hz': [0.20000000000000284, 47.5, 0.0011, 0.11],
 '315_Hz': [0.6999999999999957, 107.0, 0.0033, 0.33],
 '400_Hz': [0.4000000000000057, 131.5, 0.0077, 0.77],
 '500_Hz': [0.3999999999999915, 148.0, 0.0105, 1.05],
 '630_Hz': [0.3999999999999915, 97.0, 0.0031, 0.31],
 '800_Hz': [0.3999999999999915, 107.0, 0.0019, 0.19],
 '1000_Hz': [0.7000000000000028, 67.0, 0.0001, 0.01],
 '1250_Hz': [1.2999999999999972, 39.0, 0.0, 0.0],
 '1600_Hz': [1.8999999999999915, 6.5, 0.0, 0.0],
 '2000_Hz': [2.8999999999999915, 9.0, 0.0, 0.0],
 '2500_Hz': [1.7000000000000028, 25.0, 0.0, 0.0],
 '3150_Hz': [0.6000000000000085, 38.0, 0.0, 0.0],
 '4000_Hz': [0.10000000000000853, 237.5, 0.4421, 44.21],
 '5000_Hz': [0.0, 239.5, 0.4633, 46.33],
 '6300_Hz': [-0.20000000000000284, 214.0, 0.1531, 15.310000000000002],
 '8000_Hz': [-0.20000000000000284, 239.5, 0.216, 21.6]}

In [14]:
wilcoxon_test

,250_Hz,315_Hz,400_Hz,500_Hz,630_Hz,800_Hz,1000_Hz,1250_Hz,1600_Hz,2000_Hz,2500_Hz,3150_Hz,4000_Hz,5000_Hz,6300_Hz,8000_Hz
Median of difference,0.2000,0.7000,0.4000,0.4000,0.4000,0.4000,0.7000,1.3,1.9,2.9,1.7,0.6,0.1000,0.0000,-0.2000,-0.200
Wilcoxon signed rank statistic,47.5000,107.0000,131.5000,148.0000,97.0000,107.0000,67.0000,39.0,6.5,9.0,25.0,38.0,237.5000,239.5000,214.0000,239.500
p value,0.0011,0.0033,0.0077,0.0105,0.0031,0.0019,0.0001,0.0,0.0,0.0,0.0,0.0,0.4421,0.4633,0.1531,0.216
p value %,0.1100,0.3300,0.7700,1.0500,0.3100,0.1900,0.0100,0.0,0.0,0.0,0.0,0.0,44.2100,46.3300,15.3100,21.600


In [15]:
# write to xlsx file
wilcoxon_test.to_excel("/media/guido/LACIE/Cingle_Guido/Analysis_results/analysis_diff_max_vs_firstfit_BH5P.xlsx",
                         sheet_name='diff_max_vs_firstfit')  